<a href="https://colab.research.google.com/github/asmaatefomran/generative-ai-tasks/blob/main/Task5_rag_system_using_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 5: Build a RAG System Using LangChain**

## Objective

Build a simple Retrieval-Augmented Generation (RAG) system using
LangChain to answer questions based on a PDF document.


In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma pypdf sentence-transformers langchain-groq


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137

Imports


In [2]:
import os

from google.colab import userdata, files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq


/tmp/ipykernel_3231/1533257052.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Load Groq API key

In [3]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


Upload your PDF

In [4]:
uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Loaded:", pdf_file)


Saving Coding-Challenge-Full-Stack-Developer (1).pdf to Coding-Challenge-Full-Stack-Developer (1).pdf
Loaded: Coding-Challenge-Full-Stack-Developer (1).pdf


Load the PDF

In [5]:
loader = PyPDFLoader(pdf_file)

documents = loader.load()

print("Pages:", len(documents))


Pages: 4


Split the document

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))


Chunks: 16


Create embeddings

In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Create vector store

In [8]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


Create the LangChain LLM

In [9]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)


Simple LangChain RAG

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

If the answer is not in the context, say:
"I could not find the answer in the document."

Context:
{context}

Question:
{question}

Answer:
""")


In [11]:
def rag(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    chain = prompt | llm | StrOutputParser()

    return chain.invoke({
        "context": context,
        "question": question
    })


Testing

In [13]:
while True:

    question = input("\nAsk a question (or type 'exit'): ")

    if question.lower() == "exit":
        break

    print("\nAnswer:")
    print(rag(question))



Ask a question (or type 'exit'): What is the main topic of the document?

Answer:
The document explains how to structure a learning program—defining steps, groups, ordering, choice rules, and prerequisites for building and managing the program’s curriculum.

Ask a question (or type 'exit'): What are the key findings discussed in the document?

Answer:
The document highlights several key outcomes:

- **API functionality** – The program‑retrieval endpoint (`GET /programs/:id`) and the validation endpoint (`POST /programs/:id/validate`) are in place and return the expected data (full structure, validity status, impossible prerequisites, and reachability warnings).  
- **Reachability‑warning logic** – The logic for generating reachability warnings (Part 2) has been implemented.  
- **Comprehensive test coverage** – A test suite (Part 3) has been written that covers all required scenarios, including:
  * Successful validation of the full Computer Science scenario without errors or warnings

PDF
 ↓
LangChain Loader
 ↓
LangChain Splitter
 ↓
LangChain Vector Store
 ↓
LangChain Retriever
 ↓
LangChain Prompt
 ↓
LangChain LLM
 ↓
Answer
